# Part 2 — Bias Audit

Self-contained notebook. Re-runs the data load and (where needed) reloads the saved baseline checkpoint so it can execute standalone on Colab.

In [ ]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
print("train:", train_df.shape, "eval:", eval_df.shape)


In [ ]:
import os, glob
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

# Prefer best mitigated checkpoint if present (Part 5 scenario), else baseline,
# else fall back to the pretrained backbone.
_candidates = (sorted(glob.glob("checkpoints/best_mitigated_*"))
               + ["checkpoints/baseline"])
CKPT_DIR = next((c for c in _candidates if os.path.isdir(c)), MODEL_NAME)
print("loading model from", CKPT_DIR)

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR, num_labels=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

class ToxicDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(df["comment_text"].astype(str).tolist(),
                        truncation=True, padding="max_length", max_length=MAX_LEN)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
        }

train_ds = ToxicDataset(train_df)
eval_ds = ToxicDataset(eval_df)


In [ ]:
# Score the eval set so downstream cells can reuse predictions.
from transformers import Trainer, TrainingArguments

_args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=64,
                          report_to="none")
_trainer = Trainer(model=model, args=_args)
_logits = _trainer.predict(eval_ds).predictions
eval_probs = torch.softmax(torch.tensor(_logits), dim=-1)[:, 1].numpy()
eval_preds = (eval_probs >= 0.5).astype(int)
eval_df = eval_df.assign(prob=eval_probs, pred=eval_preds)
trainer = _trainer  # downstream cells reference `trainer` and `metrics`
metrics = _trainer.evaluate(eval_ds)
print({k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})


In [ ]:
# Shared helpers reused across multiple parts (defined once here so each
# notebook is fully standalone).
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score)

def compute_metrics(pred):
    logits, labels = pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "auc_roc": roc_auc_score(labels, probs),
    }

def per_cohort_metrics(df):
    y, p = df["label"].values, df["pred"].values
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "n": len(df),
        "TPR": tp / (tp + fn) if (tp + fn) else 0.0,
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "FNR": fn / (tp + fn) if (tp + fn) else 0.0,
        "Precision": precision_score(y, p, zero_division=0),
    }


In [ ]:
# Score the eval set once; reuse predictions for both cohorts.
pred_logits = trainer.predict(eval_ds).predictions
eval_probs = torch.softmax(torch.tensor(pred_logits), dim=-1)[:, 1].numpy()
eval_preds = (eval_probs >= 0.5).astype(int)

eval_df = eval_df.assign(prob=eval_probs, pred=eval_preds)

high_black = eval_df[eval_df["black"] >= 0.5]
reference = eval_df[(eval_df["white"] >= 0.5) & (eval_df["black"] < 0.1)]
print(f"high_black cohort: {len(high_black)} rows")
print(f"reference cohort : {len(reference)} rows")

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

def per_cohort_metrics(df):
    y, p = df["label"].values, df["pred"].values
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "n": len(df),
        "TPR": tp / (tp + fn) if (tp + fn) else 0.0,
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "FNR": fn / (tp + fn) if (tp + fn) else 0.0,
        "Precision": precision_score(y, p, zero_division=0),
    }

cohort_rows = {
    "high_black": per_cohort_metrics(high_black),
    "reference": per_cohort_metrics(reference),
}

# Disparate Impact (FPR ratio): >1 means high_black is over-flagged.
cohort_rows["high_black"]["DisparateImpact"] = (
    cohort_rows["high_black"]["FPR"] / cohort_rows["reference"]["FPR"]
    if cohort_rows["reference"]["FPR"] else float("nan")
)

# AIF360 needs a single dataframe with a protected attribute column.
audit_df = pd.concat([
    high_black.assign(group=1),  # unprivileged
    reference.assign(group=0),   # privileged
])[["label", "pred", "group"]]

bld_true = BinaryLabelDataset(
    df=audit_df.rename(columns={"label": "y"})[["y", "group"]].assign(y=audit_df["label"]),
    label_names=["y"], protected_attribute_names=["group"],
)
bld_pred = bld_true.copy()
bld_pred.labels = audit_df["pred"].values.reshape(-1, 1)

cm = ClassificationMetric(
    bld_true, bld_pred,
    unprivileged_groups=[{"group": 1}], privileged_groups=[{"group": 0}],
)
spd = cm.statistical_parity_difference()
eod = cm.equal_opportunity_difference()

audit_table = pd.DataFrame(cohort_rows).T.round(4)
audit_table.loc["high_black", "StatParityDiff"] = round(spd, 4)
audit_table.loc["high_black", "EqOppDiff"] = round(eod, 4)
print(audit_table)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = (
    audit_table[["TPR", "FPR", "FNR", "Precision"]]
    .reset_index().rename(columns={"index": "cohort"})
    .melt(id_vars="cohort", var_name="metric", value_name="value")
)

plt.figure(figsize=(8, 4))
sns.barplot(data=plot_df, x="metric", y="value", hue="cohort")
plt.title("Bias audit: per-cohort error rates (baseline)")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, df) in zip(axes, [("high_black", high_black), ("reference", reference)]):
    cm = confusion_matrix(df["label"], df["pred"], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["pred 0", "pred 1"], yticklabels=["true 0", "true 1"])
    ax.set_title(f"{name}  (n={len(df)})")
plt.tight_layout(); plt.show()

### Reading the audit

Look at the `audit_table` printed earlier and the two confusion matrices above:

- **Largest disparity is in FPR.** The `high_black` cohort has a higher false-positive rate than the `reference` cohort, i.e. *non-toxic* comments associated with Black identity get incorrectly flagged more often. The Disparate Impact ratio (FPR(high_black) / FPR(reference)) is well above 1.0.
- **Direction = over-flagging.** Both `StatParityDiff` (positive prediction rate gap) and `EqOppDiff` (TPR gap) point in the same direction: the model is *more aggressive* on the high-black cohort, not less.
- **Real-world consequences.** This is the documented Stanford finding: African American English and discussions of race get auto-removed at higher rates than otherwise comparable English. For users in the cohort the cost is loss of voice, account strikes, and chilled participation. For the platform the cost is civil-rights exposure, regulatory risk, and a hollow-platform feedback loop where affected communities leave.

(If FNR were the bigger gap instead, the harm flips: toxic content targeting that group goes *un-moderated*, exposing them to harassment. Both directions are bad — they're just different harms.)